[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C39_Distributed_Training_Course/04_checkpointing_faulttol/04_checkpointing_faulttol.ipynb)

# 04 · Checkpoint 与容错（用 numpy 模拟）

目标：在**单进程**里用 numpy + stdlib 模拟「多 rank + 文件系统」，亲手把容错的核心机制做出来并用 `assert` 验证：
**MTBF 数学 → 分片存取 → 原子写入 → 异步重叠 → 精确续训（含 RNG）→ 最优频率**。

路线：MTBF/故障率 → 分片 save/load 往返 → 原子 rename（崩溃安全）→ 异步重叠开销模型 → **逐位精确续训（crown jewel）** → Young/Daly 最优间隔 → ✏️ 练习 → 📖 答案 → 🧪 真实集群胶囊 → 🔧 真实 DCP 旁注。

> 心智模型：**一个 rank = 一段 numpy 数组（它持有的分片）；持久化存储 = 一个临时目录里的文件 / 一个 dict**。我们把分布式的*结构*在单进程里展开，逻辑与真实集群一一对应。

## 1 · MTBF 数学：规模越大，越频繁地坏

单卡平均无故障时间 $T$，$N$ 张卡串联（任一坏则训练卡住），系统 MTBF $\approx T/N$。
在「故障率恒定」（泊松过程）假设下，一个时长 $w$ 的窗口里**至少发生一次故障**的概率是 $1-e^{-w/\text{MTBF}_{sys}}$。

我们把这套算清楚，并用蒙特卡洛抽样**对拍**解析公式。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def system_mtbf(single_gpu_mtbf_hours, n_gpus):
    '''N 张卡串联系统的期望故障间隔（小时）。'''
    return single_gpu_mtbf_hours / n_gpus

def prob_failure_within(window_hours, sys_mtbf_hours):
    '''泊松过程：时长 window 内至少一次故障的概率。'''
    return 1.0 - np.exp(-window_hours / sys_mtbf_hours)

T = 5 * 365 * 24            # 单卡 MTBF 取 5 年（小时）
print(f"{'卡数 N':>8s} {'系统MTBF(小时)':>16s} {'24h内挂一次的概率':>20s}")
for N in [8, 256, 1024, 16384]:
    m = system_mtbf(T, N)
    pf = prob_failure_within(24, m)
    print(f'{N:>8d} {m:>16.1f} {pf:>19.1%}')

# 对 1024 卡的系统 MTBF 做一个手算对拍
assert abs(system_mtbf(T, 1024) - T/1024) < 1e-9
assert abs(system_mtbf(T, 1024) - 42.78) < 0.1   # 43800/1024 ≈ 42.8 小时
print('\n✅ 1024 卡每 ~43 小时挂一次；16384 卡几乎每天必挂数次 —— 没有 checkpoint 跑不完')

**蒙特卡洛对拍**：直接模拟「每张卡的故障时刻服从指数分布」，统计窗口内系统是否发生过故障，看经验概率是否逼近解析公式 $1-e^{-w/M_{sys}}$。

In [ ]:
def montecarlo_failure_prob(single_mtbf, n_gpus, window, trials=20000, seed=1):
    g = np.random.default_rng(seed)
    # 每次试验：N 张卡，各自第一次故障时刻 ~ Exponential(mean=single_mtbf)
    first_fail = g.exponential(single_mtbf, size=(trials, n_gpus)).min(axis=1)
    return float((first_fail <= window).mean())

N, w = 256, 24.0
analytic = prob_failure_within(w, system_mtbf(T, N))
empirical = montecarlo_failure_prob(T, N, w)
print(f'解析公式 = {analytic:.4f}   蒙特卡洛 = {empirical:.4f}')
assert abs(analytic - empirical) < 0.01, '解析与抽样应当吻合'
print('✅ 串联系统故障概率：解析 ≈ 抽样，1-exp(-w/MTBF_sys) 成立')

## 2 · 分片 save / load 往返

大模型没有任何一张卡装得下完整状态，所以每个 rank **只存自己那一片**，并行写 W 个文件 + 一份元数据。

我们用一个临时目录当「持久化存储」：把一个全局参数向量切成 W 片，各 rank 各存各的，再独立地全部读回、按全局偏移拼接，**对拍**原始向量。

In [ ]:
import os, tempfile, pickle

def shard_indices(n, world):
    '''把长度 n 的全局张量尽量均匀切成 world 段，返回每段 (start, end)。'''
    base, extra = divmod(n, world)
    bounds, s = [], 0
    for r in range(world):
        ln = base + (1 if r < extra else 0)   # 前 extra 段多 1 个，处理不整除
        bounds.append((s, s + ln)); s += ln
    return bounds

def save_sharded(store_dir, global_tensor, world, step):
    '''每个 rank 存自己那片：文件里带上「全局偏移」(re-shard 的关键，第7节/练习4用)。'''
    os.makedirs(store_dir, exist_ok=True)
    for r, (a, b) in enumerate(shard_indices(len(global_tensor), world)):
        payload = dict(rank=r, world=world, step=step, offset=a,
                       shard=np.array(global_tensor[a:b]))
        with open(os.path.join(store_dir, f'shard_{r}.pkl'), 'wb') as f:
            pickle.dump(payload, f)

def load_sharded(store_dir, world):
    '''各 rank 读回各自分片，按全局 offset 拼回完整张量。'''
    parts = []
    for r in range(world):
        with open(os.path.join(store_dir, f'shard_{r}.pkl'), 'rb') as f:
            parts.append(pickle.load(f))
    parts.sort(key=lambda d: d['offset'])
    return np.concatenate([d['shard'] for d in parts]), parts

params = rng.standard_normal(100).astype(np.float64)
d = tempfile.mkdtemp()
save_sharded(d, params, world=8, step=1000)
restored, parts = load_sharded(d, world=8)
print(f'切成 8 片：各片长度 = {[len(p["shard"]) for p in parts]}')
assert restored.shape == params.shape
assert np.array_equal(restored, params), '分片往返必须逐位无损'
assert all(p['step'] == 1000 for p in parts)
print('✅ 分片 save/load 往返逐位一致；8 个文件并行写，无单卡 OOM')

## 3 · 原子写入：写到一半崩溃也不丢

**反模式**：直接覆盖唯一的 checkpoint，写一半崩 → 新旧俱毁。
**正确**：写 `.tmp` → `os.rename` 原子替换。POSIX 保证 rename 要么全成、要么没发生。

我们模拟一次「写到一半崩溃」，验证：原子套路下**旧 checkpoint 依然完好可恢复**，而覆盖式写则全毁。

In [ ]:
class CrashDuringWrite(Exception):
    pass

def unsafe_overwrite(path, data, crash_at=None):
    '''反模式：直接截断目标文件分块写；crash_at 处抛异常模拟崩溃。'''
    with open(path, 'wb') as f:
        for i, chunk in enumerate(data):
            if crash_at is not None and i == crash_at:
                raise CrashDuringWrite('boom (overwrite)')
            f.write(chunk); f.flush()

def atomic_write(path, data, crash_at=None):
    '''正确：写临时文件→fsync→原子 rename。崩在写 .tmp 阶段则正式文件分毫未动。'''
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f:
        for i, chunk in enumerate(data):
            if crash_at is not None and i == crash_at:
                raise CrashDuringWrite('boom (atomic, only .tmp affected)')
            f.write(chunk); f.flush()
        os.fsync(f.fileno())          # 确保数据真正落盘，不只在 OS 缓存
    os.rename(tmp, path)              # 原子替换：要么全成要么没发生

d = tempfile.mkdtemp(); path = os.path.join(d, 'ckpt.bin')
good = [b'GOOD-CKPT-v1' for _ in range(4)]
new  = [b'NEW-CKPT-v2!' for _ in range(4)]

# 先放一个完好的旧 checkpoint
atomic_write(path, good)
assert open(path,'rb').read() == b''.join(good)
print('已写入完好旧 checkpoint:', open(path,'rb').read()[:12], '...')

In [ ]:
# 情形 A：覆盖式写新版，写到一半崩溃
try:
    unsafe_overwrite(path, new, crash_at=2)
except CrashDuringWrite as e:
    print('覆盖式写崩溃:', e)
corrupted = open(path,'rb').read()
print('  崩溃后正式文件内容:', corrupted, ' <- 既不是旧版也不是新版 = 损坏！')
assert corrupted != b''.join(good) and corrupted != b''.join(new)

# 重置旧 checkpoint，改用原子写新版，写到一半崩溃
atomic_write(path, good)
try:
    atomic_write(path, new, crash_at=2)
except CrashDuringWrite as e:
    print('原子写崩溃:', e)
survived = open(path,'rb').read()
print('  崩溃后正式文件内容:', survived, ' <- 旧 checkpoint 完好无损！')
assert survived == b''.join(good), '原子写崩溃后旧版必须完好'
assert os.path.exists(path + '.tmp'), '半截数据只在 .tmp 里（可安全删除）'
print('✅ 原子 rename：崩溃绝不损坏正式 checkpoint；覆盖式写则新旧俱毁')

## 4 · 异步 / 重叠 checkpoint：把写盘藏到训练背后

拷贝到 CPU（HBM→主存）很快，落盘很慢。同步开销 = `copy + write`；
异步把落盘丢给后台线程、训练立刻继续，**有效开销 ≈ copy**（只要落盘在下次 checkpoint 前完成）。

我们用一个**时间记账模型**模拟训练步与 checkpoint，比较同步 vs 异步暴露给训练循环的有效开销。

In [ ]:
def run_training(n_steps, ckpt_every, t_step, t_copy, t_write, mode):
    '''返回 (训练总时长, checkpoint 暴露给主循环的有效开销之和)。
       mode='none'|'sync'|'async'。async: 落盘与后续训练重叠，只在 copy 期间停。'''
    wall = 0.0; ckpt_overhead = 0.0
    bg_done_at = 0.0                 # 后台落盘完成时刻（async 用）
    for step in range(1, n_steps + 1):
        wall += t_step               # 一步训练
        if ckpt_every and step % ckpt_every == 0:
            if mode == 'sync':
                wall += t_copy + t_write; ckpt_overhead += t_copy + t_write
            elif mode == 'async':
                # 必须等上一次后台落盘完成，否则堆积（退化为同步）
                if wall < bg_done_at:
                    stall = bg_done_at - wall
                    wall += stall; ckpt_overhead += stall
                wall += t_copy; ckpt_overhead += t_copy   # 只停拷贝
                bg_done_at = wall + t_write                # 落盘在后台进行
    return wall, ckpt_overhead

kw = dict(n_steps=200, ckpt_every=20, t_step=1.0, t_copy=0.3, t_write=5.0)
_, ov_sync  = run_training(mode='sync',  **kw)
_, ov_async = run_training(mode='async', **kw)
print(f'同步 checkpoint 有效开销 = {ov_sync:.1f}  (= 10 次 × (copy0.3+write5.0))')
print(f'异步 checkpoint 有效开销 = {ov_async:.1f}  (≈ 10 次 × copy0.3)')
assert abs(ov_sync - 10 * 5.3) < 1e-9
assert ov_async < ov_sync and ov_async <= 10 * 0.3 + 1e-9
print(f'✅ 异步把有效开销从 {ov_sync:.1f} 压到 {ov_async:.1f}（≈只剩 copy）—— 落盘藏到训练背后')

## 5 · 逐位精确续训：RNG 状态是关键（crown jewel）

要让「中断→恢复」后的训练轨迹和**从未中断**逐位相同，必须保存并恢复 **RNG 内部状态**——否则 dropout/数据增强/采样的随机序列错位，轨迹悄悄发散。

我们训一个玩具：每步用随机数生成一个「梯度」更新参数（模拟 dropout/增强带来的随机性）。对照三种 resume：① 不存 RNG、② 存 RNG。验证只有存了 RNG 才能与不中断轨迹**逐位一致**。

> 关键 API：`g.bit_generator.state` 可读出/写回 numpy `Generator` 的完整内部状态。

In [ ]:
def rng_get_state(g):
    '''读出 Generator 的完整内部状态（可 pickle、可写回）。'''
    import copy
    return copy.deepcopy(g.bit_generator.state)

def rng_set_state(g, state):
    import copy
    g.bit_generator.state = copy.deepcopy(state)

def train_steps(params, g, n_steps, lr=0.1):
    '''每步：用 RNG 产生一个随机梯度（模拟 dropout/增强的随机性），更新参数。
       返回更新后的 params（原地修改 g 的 RNG 状态）。'''
    p = params.copy()
    for _ in range(n_steps):
        grad = g.standard_normal(p.shape)     # 消耗随机数 -> RNG 状态前进
        p = p - lr * grad
    return p

# 基准：一口气训 10 步，绝不中断
p0 = np.zeros(5)
g_ref = np.random.default_rng(1234)
p_uninterrupted = train_steps(p0, g_ref, 10)
print('不中断 10 步的最终参数:', np.round(p_uninterrupted, 4))

In [ ]:
# 模拟：训 5 步 -> 存 checkpoint -> 崩溃 -> 恢复 -> 再训 5 步
# 情形 ①：只存参数，不存 RNG（恢复时重新 seed / 用新 RNG）
g_a = np.random.default_rng(1234)
p_mid_a = train_steps(p0, g_a, 5)                 # 训 5 步并存档
ckpt_no_rng = dict(params=p_mid_a.copy(), step=5) # 漏存 RNG！
# ...崩溃、重启... 用一个「新」RNG 续训（常见 bug）
g_a2 = np.random.default_rng(1234)               # 即便同 seed，也已被前 5 步「用过」的状态对不上
p_resume_no_rng = train_steps(ckpt_no_rng['params'], g_a2, 5)

# 情形 ②：参数 + RNG 状态都存
g_b = np.random.default_rng(1234)
p_mid_b = train_steps(p0, g_b, 5)
ckpt_full = dict(params=p_mid_b.copy(), step=5, rng=rng_get_state(g_b))  # 完整！
# ...崩溃、重启... 恢复 RNG 状态再续训
g_b2 = np.random.default_rng()                   # 任意新 Generator
rng_set_state(g_b2, ckpt_full['rng'])            # 把 RNG 拨回存档那一刻
p_resume_full = train_steps(ckpt_full['params'], g_b2, 5)

print('不中断           :', np.round(p_uninterrupted, 6))
print('恢复(漏存RNG)    :', np.round(p_resume_no_rng, 6), '<- 偏离！')
print('恢复(存了RNG)    :', np.round(p_resume_full, 6), '<- 逐位一致')
assert not np.allclose(p_resume_no_rng, p_uninterrupted), '漏存 RNG 必然偏离'
assert np.array_equal(p_resume_full, p_uninterrupted), '存了 RNG 才能逐位精确续训'
print('\n✅ crown jewel：只有保存并恢复 RNG 状态，resume 才与不中断轨迹逐位相同')

## 6 · Young/Daly 最优 checkpoint 间隔

存太勤 → checkpoint 开销大；存太疏 → 故障回滚损失大。把两者之和对间隔 $\tau$ 求极小：

$$\tau^{*} \approx \sqrt{2\,\delta\,M}, \quad \delta=\text{单次开销},\ M=\text{系统 MTBF}$$

我们建一个简单的总开销模型（每步开销摊销 + 期望回滚损失），用**网格搜索**找到最优 $\tau$，验证它落在 $\sqrt{2\delta M}$ 附近。

In [ ]:
def overhead_per_unit(tau, delta, M):
    '''单位时间的额外开销 ≈ checkpoint 摊销(delta/tau) + 期望回滚损失(tau/(2M))。
       (一阶模型：每 tau 存一次 -> 摊销 delta/tau；故障平均回滚半个区间 tau/2，频率 1/M。)'''
    return delta / tau + tau / (2 * M)

def young_daly(delta, M):
    return np.sqrt(2 * delta * M)

delta, M = 0.5, 200.0                  # 单次 checkpoint 0.5h；系统 MTBF 200h
taus = np.linspace(1.0, 80.0, 8000)
costs = np.array([overhead_per_unit(t, delta, M) for t in taus])
tau_grid = taus[np.argmin(costs)]
tau_formula = young_daly(delta, M)
print(f'网格搜索最优 τ = {tau_grid:.2f} h')
print(f'Young/Daly 公式 = {tau_formula:.2f} h  (= sqrt(2·0.5·200))')
assert abs(tau_grid - tau_formula) < 0.5, '网格最优应逼近闭式解'
# 单调性：delta 越小 / M 越小，最优间隔越小（越该多存）
assert young_daly(0.1, M) < young_daly(0.5, M)
assert young_daly(delta, 50) < young_daly(delta, 200)
print('✅ 网格最优 ≈ sqrt(2δM)；δ 小(异步)或 M 小(集群大)都该存得更勤')

---
## ✏️ 练习 1：不整除的分片往返

现实中张量长度常不是 world 的整数倍。实现 `save_then_load(global_tensor, world)`：
用第 2 节的 `shard_indices` 切片（前若干片多 1 个元素），存进一个 dict（key=rank），再读回拼接。

要求：返回拼接后的张量；对**任意** `world`（含不整除）都逐位还原原张量。

In [ ]:
def save_then_load(global_tensor, world):
    n = len(global_tensor)
    store = {}   # 模拟持久化存储：rank -> (offset, shard)
    # TODO: 用 shard_indices(n, world) 得到每片 (a,b)；store[r]=(a, global_tensor[a:b].copy())
    #       再按 offset 排序、concatenate 还原
    raise NotImplementedError
    return restored

In [ ]:
# —— 练习 1 自测 ——
for n, world in [(100, 8), (10, 3), (7, 4), (5, 5), (13, 6)]:
    x = rng.standard_normal(n)
    r = save_then_load(x, world)
    assert r.shape == x.shape, f'形状不对 n={n} world={world}'
    assert np.array_equal(r, x), f'分片往返必须逐位无损 n={n} world={world}'
print('✅ 练习 1 通过：不整除分片也能逐位还原')

## ✏️ 练习 2：异步 checkpoint 的「堆积」陷阱

异步只有在「后台落盘能在下次 checkpoint 前完成」时才划算；否则两次落盘堆积、退化为同步。

实现 `async_effective_overhead(n_ckpts, gap, t_copy, t_write)`：`gap` = 相邻两次 checkpoint 之间的训练时间。
若 `t_write <= gap`（落盘来得及）→ 每次只暴露 `t_copy`；否则每次额外暴露 `t_write - gap` 的堆积停顿。
返回总有效开销。

In [ ]:
def async_effective_overhead(n_ckpts, gap, t_copy, t_write):
    # TODO: 每次暴露 t_copy；若 t_write > gap，再加 (t_write - gap) 的堆积停顿
    #       返回 n_ckpts 次的总有效开销
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 落盘来得及：只暴露 copy
ok = async_effective_overhead(10, gap=20.0, t_copy=0.3, t_write=5.0)
assert abs(ok - 10 * 0.3) < 1e-9
# 落盘来不及（gap 太小）：每次额外堆积 (t_write-gap)
bad = async_effective_overhead(10, gap=2.0, t_copy=0.3, t_write=5.0)
assert abs(bad - 10 * (0.3 + (5.0 - 2.0))) < 1e-9
assert bad > ok
print(f'来得及={ok:.1f}  来不及(堆积)={bad:.1f}')
print('✅ 练习 2 通过：落盘跟不上频率时，异步退化、堆积停顿出现')

## ✏️ 练习 3：一致性校验（步数对齐）

分片 checkpoint 必须所有分片同属**一个训练步**。实现 `is_consistent(shards)`：
`shards` 是 `[{'rank':r, 'step':s, ...}, ...]`。当且仅当所有分片 step 相同、且 rank 恰好是 `0..W-1` 各一份（无缺失/重复）时返回 `True`。

In [ ]:
def is_consistent(shards):
    # TODO: 检查 (a) 所有 step 相同；(b) ranks == {0,1,...,len(shards)-1}
    #       任一不满足返回 False，否则 True
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
good = [{'rank':0,'step':100}, {'rank':1,'step':100}, {'rank':2,'step':100}]
bad_step = [{'rank':0,'step':100}, {'rank':1,'step':101}, {'rank':2,'step':100}]  # rank1 多走一步
bad_miss = [{'rank':0,'step':100}, {'rank':0,'step':100}, {'rank':2,'step':100}]  # 缺 rank1、rank0 重复
assert is_consistent(good) is True
assert is_consistent(bad_step) is False, '步数不一致必须判 False'
assert is_consistent(bad_miss) is False, 'rank 缺失/重复必须判 False'
print('✅ 练习 3 通过：能抓出步数错位与分片缺失')

## ✏️ 练习 4：弹性 re-shard（W=4 → W=3）

弹性训练的核心：把按 `W_old` 存的分片**重新切**给 `W_new` 个 rank。秘诀是分片自带**全局 offset**——
先按 offset 拼回完整逻辑张量，再用 `shard_indices(n, W_new)` 重新划分。

实现 `reshard(old_shards, W_new)`：`old_shards=[{'offset':a,'shard':arr}, ...]`，返回 `W_new` 个新分片（list of dict，含新 offset 与 shard）。
**不变量**：新分片按 offset 拼接 == 旧分片按 offset 拼接。

In [ ]:
def reshard(old_shards, W_new):
    # TODO: (1) 按 offset 排序旧分片、concatenate 成完整逻辑张量 full
    #       (2) 用 shard_indices(len(full), W_new) 重新切，构造新分片 [{'offset':a,'shard':full[a:b]}]
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
full0 = rng.standard_normal(20)
# 先按 W=4 切成旧分片（带全局 offset）
old = [{'offset':a, 'shard':full0[a:b].copy()} for (a,b) in shard_indices(20, 4)]
new = reshard(old, W_new=3)
assert len(new) == 3
rebuilt_old = np.concatenate([s['shard'] for s in sorted(old, key=lambda d:d['offset'])])
rebuilt_new = np.concatenate([s['shard'] for s in sorted(new, key=lambda d:d['offset'])])
assert np.array_equal(rebuilt_old, full0)
assert np.array_equal(rebuilt_new, full0), 're-shard 必须保持拼接不变量（逐位）'
print(f'旧分片长度 {[len(s["shard"]) for s in old]} -> 新分片长度 {[len(s["shard"]) for s in new]}')
print('✅ 练习 4 通过：W=4 → W=3 re-shard，全局张量逐位不变')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def save_then_load(global_tensor, world):
    n = len(global_tensor)
    store = {}
    for r, (a, b) in enumerate(shard_indices(n, world)):
        store[r] = (a, global_tensor[a:b].copy())
    items = sorted(store.values(), key=lambda t: t[0])   # 按 offset 排序
    restored = np.concatenate([sh for _, sh in items])
    return restored

In [ ]:
# 练习 2 参考答案
def async_effective_overhead(n_ckpts, gap, t_copy, t_write):
    stall = max(0.0, t_write - gap)      # 落盘超出 gap 的部分会堆积成停顿
    return n_ckpts * (t_copy + stall)

In [ ]:
# 练习 3 参考答案
def is_consistent(shards):
    steps = {s['step'] for s in shards}
    if len(steps) != 1:
        return False
    ranks = sorted(s['rank'] for s in shards)
    return ranks == list(range(len(shards)))

In [ ]:
# 练习 4 参考答案
def reshard(old_shards, W_new):
    ordered = sorted(old_shards, key=lambda d: d['offset'])
    full = np.concatenate([s['shard'] for s in ordered])
    new = []
    for (a, b) in shard_indices(len(full), W_new):
        new.append({'offset': a, 'shard': full[a:b].copy()})
    return new

---
## 🧪 真实数据胶囊：给真实集群算最优 checkpoint 频率

用真实量级的数字，算一笔万卡训练的 checkpoint 账：单卡 MTBF、集群规模 → 系统 MTBF；单次 checkpoint 开销 → Young/Daly 最优间隔；并比较「同步 vs 异步」如何改变最优频率与故障浪费。

（纯算术，离线可跑；数字为公开报告的量级，非某次具体运行。）

In [ ]:
# 真实量级参数（公开报告的数量级；非特定某次运行）
REAL = dict(
    single_gpu_mtbf_h = 5 * 365 * 24,   # 单卡 MTBF 约 5 年
    n_gpus            = 1024,           # 集群规模
    ckpt_sync_h       = 8 / 60,         # 同步 checkpoint 约 8 分钟
    ckpt_async_h      = 0.5 / 60,       # 异步只暴露拷贝 约 30 秒
)
M = system_mtbf(REAL['single_gpu_mtbf_h'], REAL['n_gpus'])
tau_sync  = young_daly(REAL['ckpt_sync_h'],  M)
tau_async = young_daly(REAL['ckpt_async_h'], M)
print(f'系统 MTBF        = {M:.1f} h  (1024 卡, 单卡 5 年)')
print(f'最优间隔 (同步)  = {tau_sync:.2f} h   单次开销 {REAL["ckpt_sync_h"]*60:.0f} min')
print(f'最优间隔 (异步)  = {tau_async:.2f} h   单次开销 {REAL["ckpt_async_h"]*60:.0f} min')
# 异步开销更小 -> 公式给出更短的最优间隔（存得更勤）
assert tau_async < tau_sync
# 估算稳态「浪费率」≈ 单位时间额外开销（用各自最优 τ 代回一阶模型）
waste_sync  = overhead_per_unit(tau_sync,  REAL['ckpt_sync_h'],  M)
waste_async = overhead_per_unit(tau_async, REAL['ckpt_async_h'], M)
print(f'\n稳态浪费率 (同步) ≈ {waste_sync:.2%} 的算力花在 checkpoint+回滚')
print(f'稳态浪费率 (异步) ≈ {waste_async:.2%}')
assert waste_async < waste_sync
print('✅ 异步把单次开销压小 -> 可更高频 -> 故障回滚更少 -> 总浪费率更低')

**🧪 胶囊练习**：实现 `goodput(M, tau, delta, recover_h)`：估算稳态 **goodput**（有效算力占比）
= 1 − (checkpoint 摊销 `delta/tau` + 期望回滚 `tau/(2M)` + 故障恢复 `recover_h/M`)。
用它比较：把单卡 MTBF 不变、集群从 1024 扩到 8192 卡时，goodput 怎么变。

In [ ]:
def goodput(M, tau, delta, recover_h):
    # TODO: 返回 1 - (delta/tau + tau/(2*M) + recover_h/M)
    raise NotImplementedError

In [ ]:
# 自测
delta_a, recover = REAL['ckpt_async_h'], 0.25     # 异步开销 + 恢复 15 分钟
for N in [1024, 8192]:
    m = system_mtbf(REAL['single_gpu_mtbf_h'], N)
    t = young_daly(delta_a, m)
    gp = goodput(m, t, delta_a, recover)
    print(f'N={N:>5d}  系统MTBF={m:6.1f}h  最优τ={t:5.2f}h  goodput={gp:.2%}')
gp_1k = goodput(system_mtbf(REAL['single_gpu_mtbf_h'],1024), young_daly(delta_a, system_mtbf(REAL['single_gpu_mtbf_h'],1024)), delta_a, recover)
gp_8k = goodput(system_mtbf(REAL['single_gpu_mtbf_h'],8192), young_daly(delta_a, system_mtbf(REAL['single_gpu_mtbf_h'],8192)), delta_a, recover)
assert 0 < gp_8k < gp_1k <= 1.0, '集群越大 goodput 越低（故障越频繁）'
print('✅ 胶囊练习通过：规模扩大 -> 系统 MTBF 下降 -> goodput 下降（容错越重要）')

In [ ]:
# 📖 胶囊参考答案
def goodput(M, tau, delta, recover_h):
    return 1.0 - (delta / tau + tau / (2 * M) + recover_h / M)

---
## 🔧 旁注：真实的 `torch.distributed.checkpoint`（DCP）长什么样

本课在单进程里模拟的「分片 save/load + 全局 offset + re-shard」，在 PyTorch 里就是 DCP（伪代码，**本环境不跑**）：

```python
import torch.distributed.checkpoint as dcp
from torch.distributed.checkpoint.state_dict import get_state_dict, set_state_dict

# —— 保存：每个 rank 各写自己那片，并行写 + 元数据（对应我们的 save_sharded）——
state = {'model': model, 'optim': optimizer}      # 含参数+优化器分片
dcp.save(state_dict=get_state_dict(model, optimizer),
         storage_writer=dcp.FileSystemWriter('ckpt/step-1000'))

# —— 异步保存：返回 future，落盘在后台，训练立刻继续（对应第 4 节）——
future = dcp.async_save(state_dict=..., storage_writer=...)
# ... 继续训练 ...; future.result()  # 下次 checkpoint 前确认完成

# —— 加载 + 自动 re-shard：存时 8 卡、现在 4 卡也能加载（对应第 7 节/练习 4）——
dcp.load(state_dict=get_state_dict(model, optimizer),
         storage_reader=dcp.FileSystemReader('ckpt/step-1000'))
# DCP 按每片记录的全局 (offset, shape) 把数据重映射到当前的分片布局
```

对应关系：`FileSystemWriter`↔我们的临时目录、每片的全局 `(offset, shape)` 元数据↔我们 `save_sharded` 里存的 `offset`、`async_save` 的 future↔第 4 节的后台落盘、`dcp.load` 自动 re-shard↔练习 4 的 `reshard`。**弹性**则由 `torchrun --nnodes=MIN:MAX --max-restarts=K`（TorchElastic）在成员变化时重新 rendezvous。

### 小结
- 规模越大越易坏：系统 MTBF ≈ T/N，万卡每隔数小时必挂 → 没有 checkpoint 跑不完。
- checkpoint 必须存全：参数+**优化器**+**RNG**+step+数据指针+scheduler，少一样就无法逐位续训。
- **分片**绕开显存墙与串行墙；**原子 rename** 保证崩溃不损坏；**异步**把落盘藏到训练背后（开销≈copy）。
- **RNG 状态**是逐位精确续训的关键（worked 5）；最优频率 **τ\*≈√(2δM)**（便宜/易坏就多存）。
- **弹性 + re-shard**：分片自带全局 offset，故 world size 可变、加载端任意切片仍逐位不变。

下一站：**模块 05 · 编排与调试** —— rank/world 映射、straggler、扩展效率与瓶颈定位。